In [104]:
import pandas as pd
import numpy as np
import re


from unicodedata import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering, DBSCAN

from sklearn.cluster import AffinityPropagation
from sklearn.metrics.pairwise import cosine_similarity

In [105]:
path = "data/train/cleaned_train.csv"

df = pd.read_csv(path)

# Clusterización de versiones


## Experimentos para un modelo especificado

In [106]:
brand = "Chevrolet"
model = "Tracker"

version_df = df[(df["Marca"] == brand) & (df["Modelo"] == model)].reset_index(drop=True)

### Limpieza y preprocesamiento de versiones

Este código se encarga de limpiar y normalizar los nombres de las versiones de vehículos para facilitar su posterior análisis y agrupación:

**Función** `clean_version()`
- **Normalización de caracteres**: Convierte texto a minúsculas y elimina acentos/caracteres especiales usando Unicode NFKD
- **Filtrado de caracteres**: Solo mantiene letras, números y espacios, eliminando símbolos y puntuación
- **Limpieza de espacios**: Normaliza espacios múltiples a espacios únicos y elimina espacios al inicio/final
- **Manejo de valores nulos**: Convierte valores `NaN` a strings vacíos


In [107]:
def clean_version(s: str) -> str:
    s = "" if pd.isna(s) else s
    s = normalize("NFKD", s.lower()).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

version_df["version_clean"] = version_df["Versión"].apply(clean_version)
version_df = version_df[version_df["version_clean"] != ""].reset_index(drop=True)

print("Número de versiones únicas:", version_df["version_clean"].nunique())

print("\nPrimeras 5 versiones únicas:")
print(version_df["version_clean"].value_counts()[:5])

Número de versiones únicas: 28

Primeras 5 versiones únicas:
version_clean
ltz              717
premier          143
rs                48
t premier l20      7
lt                 7
Name: count, dtype: int64


### HAC y TF-IDF

In [108]:
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"])

non_zero_mask = np.array(X.sum(axis=1)).flatten() > 0
X_filtered = X[non_zero_mask].toarray()
taos_df_filtered = version_df[non_zero_mask].copy()

model = AgglomerativeClustering(
    metric="cosine",
    linkage="average",
    distance_threshold=0.8,
    n_clusters=None,
)

# Fit on filtered data
clusters_filtered = model.fit_predict(X_filtered)

# Assign clusters back to original dataframe
version_df["cluster"] = -1  # Default for zero vectors
version_df.loc[non_zero_mask, "cluster"] = clusters_filtered

In [109]:
canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

summary

,cluster,version_canon,count
9,8,Ltz,717
1,0,Premier,167
0,-1,Rs,55
10,9,Suv Chevrolet Tracker RS Nafta 6 Velocidades,10
3,2,T LTZ,6
4,3,argentina,6
2,1,LTZ PLUS L13,3
5,4,LTZ L13,3
6,5,Ltz+ motor Ecotec,1
7,6,MY24,1


### Affinity propagation

In [110]:
S = cosine_similarity(X)

pref = np.median(S) - 0.05
ap = AffinityPropagation(affinity="precomputed",
                         damping=0.95,
                         preference=pref,
                         max_iter=500,
                         convergence_iter=16)

version_df["cluster"] = ap.fit_predict(S)


canon = version_df.groupby("cluster")["Versión"].agg(lambda x: x.value_counts().idxmax())
version_df["version_canon"] = version_df["cluster"].map(canon)

summary = (
    version_df.groupby(["cluster", "version_canon"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
summary


,cluster,version_canon,count
2,2,Ltz,723
1,1,Premier,143
11,11,LTZ Premier,13
35,35,T PREMIER L20,13
0,0,Suv Chevrolet Tracker RS Nafta 6 Velocidades,10
...,...,...,...
27,27,Rs,1
28,28,Rs,1
29,29,Rs,1
30,30,Rs,1


### DBSCAN

- Leer `CountVectorizer` en [Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) (podemos jugar con los argumentos)

In [111]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 5))
X = vectorizer.fit_transform(version_df["version_clean"]).toarray()

print(X.shape)

(970, 405)


In [112]:
n_versions = version_df["version_clean"].nunique()

db = DBSCAN(eps=0.3, min_samples=max(2, n_versions // 4), metric="cosine")
version_df["cluster"] = db.fit_predict(X)

# canónico solo para clusters válidos (≠ –1)
canon = (
    version_df[version_df["cluster"] != -1]
      .groupby("cluster")["Versión"]
      .agg(lambda s: s.value_counts().idxmax())
)
version_df["version_canon"] = version_df["cluster"].map(canon)

# todo el ruido recibe una misma etiqueta
version_df.loc[version_df["cluster"] == -1, "version_canon"] = "Unassigned"

summary = (
    version_df.groupby(["cluster", "version_canon"])
              .size()
              .reset_index(name="count")
              .sort_values("count", ascending=False)
)
print(summary)
print("\nClusters densos:", canon.size,
      "  Etiquetas finales:", version_df["version_canon"].nunique())


   cluster                                 version_canon  count
1        0                                           Ltz    717
3        2                                       Premier    166
0       -1                                    Unassigned     77
2        1  Suv Chevrolet Tracker RS Nafta 6 Velocidades     10

Clusters densos: 3   Etiquetas finales: 4


Asignar cada muestra al centoride al que pertenece

In [113]:
version_df["Versión"] = version_df["version_canon"]
version_df.drop(columns=["cluster", "version_canon"], inplace=True)

print("Versiones únicas tras el clustering:",
      version_df["Versión"].nunique())
print("Etiquetas:", version_df["Versión"].unique())


Versiones únicas tras el clustering: 4
Etiquetas: ['Ltz' 'Suv Chevrolet Tracker RS Nafta 6 Velocidades' 'Premier'
 'Unassigned']


# Compartir features entre cada versión

In [114]:
def fill_by_mode(df, col, keys=("Marca", "Modelo", "Versión")):
    mode_vals = (
        df.groupby(list(keys))[col]
          .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
    )
    df[col] = df[col].fillna(mode_vals)

cols_to_fill = ["cv", "Motor", "Tracción", "Turbo"]      # agrega o quita las que quieras

for c in cols_to_fill:
    if c in version_df.columns:
        fill_by_mode(version_df, c)

print("Faltantes después de imputar:")
print(version_df[cols_to_fill].isna().sum())

Faltantes después de imputar:
cv           0
Motor        0
Tracción    10
Turbo        0
dtype: int64
